In [1]:
# ==========================================================================
# INSTALAÇÃO
# ==========================================================================
!pip install -q librosa soundfile noisereduce matplotlib pandas

# ==========================================================================
# IMPORTS
# ==========================================================================
import os, zipfile
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import uniform_filter
from google.colab import files

# ==========================================================================
# UPLOAD
# ==========================================================================
print(">>> Faça upload dos áudios RUIDOSOS (.wav)")
uploaded_noisy = files.upload()
noisy_files = list(uploaded_noisy.keys())
print("Áudios ruidosos:", noisy_files)

print("\n>>> Faça upload dos áudios LIMPOS de referência (.wav)")
uploaded_clean = files.upload()
clean_files = list(uploaded_clean.keys())
print("Áudios limpos:", clean_files)

# ==========================================================================
# PASTA DE SAÍDA
# ==========================================================================
OUT_DIR = "/content/denoised"
os.makedirs(OUT_DIR, exist_ok=True)

# ==========================================================================
# FILTRO DE WIENER
# ==========================================================================
def wiener_filter(filename):
    y, sr = librosa.load(filename, sr=None)
    n_fft = 1024
    hop_length = n_fft // 4
    win_length = n_fft

    Y = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    Y_mag   = np.abs(Y)
    Y_phase = np.angle(Y)
    Y_power = Y_mag ** 2
    freqs   = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    noise_end_sec = min(20.0, (len(y) / sr) / 4.0)
    f_end = max(int(noise_end_sec * sr / hop_length), 1)
    Pn = np.mean(Y_power[:, :f_end], axis=1, keepdims=True)
    Pn = np.maximum(Pn, 1e-12)

    alpha = 0.94
    eps   = 1e-10

    floor = np.interp(
        freqs,
        [0,    100,  220,  500,  1500, 3000, 6000, 12000],
        [0.35, 0.25, 0.28, 0.15, 0.12, 0.10, 0.08, 0.08]
    )
    beta = np.ones_like(freqs)
    beta[freqs >= 1000] = 1.5
    beta[freqs >= 4000] = 2.0

    H = np.ones_like(Y_power)
    gamma_prev = np.ones_like(Y_power)

    for t in range(Y_power.shape[1]):
        gamma = Y_power[:, t] / (Pn[:, 0] + eps)
        if t == 0:
            xi = np.maximum(gamma - beta, 0)
        else:
            xi = alpha * (H[:, t-1]**2 * gamma_prev[:, t-1]) + \
                 (1 - alpha) * np.maximum(gamma - beta, 0)
        H[:, t] = xi / (1 + xi)
        gamma_prev[:, t] = gamma

    H = np.maximum(H, floor[:, None])
    H = uniform_filter(H, size=(3, 5))

    Y_clean = H * Y_mag * np.exp(1j * Y_phase)
    y_wiener = librosa.istft(Y_clean, hop_length=hop_length, length=len(y))

    peak = np.max(np.abs(y_wiener))
    if peak > 1.0:
        y_wiener = y_wiener / peak * 0.99

    return y, y_wiener, sr


# ==========================================================================
# MÉTRICAS
# ==========================================================================
def _align(ref, est):
    n = min(len(ref), len(est))
    return ref[:n], est[:n]

def snr(ref, est):
    ref, est = _align(ref, est)
    noise = ref - est
    num = np.sum(ref**2)
    den = np.sum(noise**2) + 1e-12
    return 10 * np.log10(num / den)

def snr_seg(ref, est, sr, frame_len=0.02, hop=0.01):
    ref, est = _align(ref, est)
    fl = int(frame_len * sr); hp = int(hop * sr)
    if fl <= 0 or len(ref) < fl:
        return snr(ref, est)
    vals = []
    for s in range(0, len(ref) - fl, hp):
        r = ref[s:s+fl]; e = est[s:s+fl]
        noise = r - e
        num = np.sum(r**2); den = np.sum(noise**2) + 1e-12
        v = 10 * np.log10(num / den)
        if np.isfinite(v):
            vals.append(v)
    return float(np.mean(vals)) if vals else np.nan

def mse(ref, est):
    ref, est = _align(ref, est)
    return float(np.mean((ref - est) ** 2))

def rmse(ref, est):
    return float(np.sqrt(mse(ref, est)))

def sdr(ref, est):
    ref, est = _align(ref, est)
    ref = ref.astype(np.float64); est = est.astype(np.float64)
    denom = np.dot(ref, ref) + 1e-12
    alpha = np.dot(ref, est) / denom
    s_target = alpha * ref
    e_res = est - s_target
    num = np.sum(s_target ** 2)
    den = np.sum(e_res ** 2) + 1e-12
    return 10 * np.log10(num / den)

def si_sdr(ref, est):
    return sdr(ref, est)

def pearson(ref, est):
    ref, est = _align(ref, est)
    ref = ref.astype(np.float64); est = est.astype(np.float64)
    if np.std(ref) < 1e-12 or np.std(est) < 1e-12:
        return np.nan
    return float(np.corrcoef(ref, est)[0, 1])


# ==========================================================================
# MATCH RUIDOSO ↔ LIMPO
# ==========================================================================
def encontrar_par(noisy_name, clean_files):
    base = noisy_name.rsplit('.', 1)[0]
    candidates = [
        base.replace('chuva', 'limpo'),
        base.replace('ruido', 'limpo'),
        base.replace('ruído', 'limpo'),
        base.replace('_noisy', '_clean'),
        base.replace('_noise', '_clean'),
        base.replace('_n', '_l'),
    ]
    for c in clean_files:
        stem = c.rsplit('.', 1)[0]
        if stem in candidates or stem == base:
            return c
    prefix = base.split('_')[0]
    for c in clean_files:
        if c.rsplit('.', 1)[0].split('_')[0] == prefix:
            return c
    return None


# ==========================================================================
# FUNÇÃO DE IMPRESSÃO DA TABELA (formato solicitado)
# ==========================================================================
ORDEM_METRICAS = [
    ("SNR (dB)",      "SNR Ruidoso (dB)",   "SNR Wiener (dB)"),
    ("SNRseg (dB)",   "SNRseg Ruidoso (dB)","SNRseg Wiener (dB)"),
    ("MSE",           "MSE Ruidoso",        "MSE Wiener"),
    ("RMSE",          "RMSE Ruidoso",       "RMSE Wiener"),
    ("SDR (dB)",      "SDR Ruidoso (dB)",   "SDR Wiener (dB)"),
    ("SI-SDR (dB)",   "SI-SDR Ruidoso (dB)","SI-SDR Wiener (dB)"),
    ("Pearson (r)",   "Pearson Ruidoso",    "Pearson Wiener"),
]

def _fmt(v):
    """Formata com 4 casas e sinal explícito no ganho (usado à parte)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "   nan"
    return f"{v:>10.4f}"

def _fmt_gain(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "   nan"
    sign = "+" if v >= 0 else "-"
    return f"{sign}{abs(v):>9.4f}"

def imprimir_tabela(titulo, linhas):
    """
    linhas: lista de tuplas (nome_metrica, valor_ruidoso, valor_processado)
    """
    print("\n" + "=" * 80)
    print(titulo)
    print("=" * 80)
    print(f"{'Métrica':<28}{'Ruidoso':>14}{'Processado':>18}{'Ganho':>15}")
    print("=" * 80)
    for nome, r, p in linhas:
        g = (p - r) if (r is not None and p is not None
                        and not (isinstance(r, float) and np.isnan(r))
                        and not (isinstance(p, float) and np.isnan(p))) else np.nan
        print(f"{nome:<28}{_fmt(r):>12}{_fmt(p):>16}{_fmt_gain(g):>15}")
    print("=" * 80)


# ==========================================================================
# PROCESSAMENTO
# ==========================================================================
resultados = []
caminhos_salvos = []

for noisy_name in noisy_files:
    clean_name = encontrar_par(noisy_name, clean_files)
    if clean_name is None:
        print(f"[AVISO] Sem referência limpa para {noisy_name}. Pulando.")
        continue

    print(f"\n>>> Processando: {noisy_name}  (ref: {clean_name})")

    y_ref, sr_ref = librosa.load(clean_name, sr=None)
    y_noisy, y_wiener, sr = wiener_filter(noisy_name)

    if sr_ref != sr:
        y_ref = librosa.resample(y_ref, orig_sr=sr_ref, target_sr=sr)

    n = min(len(y_noisy), len(y_ref), len(y_wiener))
    y_noisy  = y_noisy[:n]
    y_wiener = y_wiener[:n]
    y_ref    = y_ref[:n]

    # ----- SALVAR -----
    out_name = noisy_name.rsplit('.', 1)[0] + '_wiener.wav'
    out_path = os.path.join(OUT_DIR, out_name)
    sf.write(out_path, y_wiener, sr)
    caminhos_salvos.append(out_path)
    print(f"    💾 Salvo: {out_path}")

    # ----- MÉTRICAS -----
    row = {
        'Áudio'              : noisy_name,
        'SNR Ruidoso (dB)'   : snr(y_ref, y_noisy),
        'SNRseg Ruidoso (dB)': snr_seg(y_ref, y_noisy, sr),
        'MSE Ruidoso'        : mse(y_ref, y_noisy),
        'RMSE Ruidoso'       : rmse(y_ref, y_noisy),
        'SDR Ruidoso (dB)'   : sdr(y_ref, y_noisy),
        'SI-SDR Ruidoso (dB)': si_sdr(y_ref, y_noisy),
        'Pearson Ruidoso'    : pearson(y_ref, y_noisy),
        'SNR Wiener (dB)'    : snr(y_ref, y_wiener),
        'SNRseg Wiener (dB)' : snr_seg(y_ref, y_wiener, sr),
        'MSE Wiener'         : mse(y_ref, y_wiener),
        'RMSE Wiener'        : rmse(y_ref, y_wiener),
        'SDR Wiener (dB)'    : sdr(y_ref, y_wiener),
        'SI-SDR Wiener (dB)' : si_sdr(y_ref, y_wiener),
        'Pearson Wiener'     : pearson(y_ref, y_wiener),
    }
    resultados.append(row)

    # ----- TABELA INDIVIDUAL -----
    linhas_individuais = [
        (nome, row[col_r], row[col_p])
        for nome, col_r, col_p in ORDEM_METRICAS
    ]
    imprimir_tabela(f"ÁUDIO: {noisy_name}   (ref: {clean_name})", linhas_individuais)

    # ----- GRÁFICO -----
    fig, axes = plt.subplots(3, 1, figsize=(15, 10))
    S_noisy = librosa.amplitude_to_db(np.abs(librosa.stft(y_noisy)),  ref=np.max)
    S_wien  = librosa.amplitude_to_db(np.abs(librosa.stft(y_wiener)), ref=np.max)
    S_ref   = librosa.amplitude_to_db(np.abs(librosa.stft(y_ref)),    ref=np.max)

    img0 = librosa.display.specshow(S_noisy, sr=sr, x_axis='time',
                                    y_axis='log', cmap='viridis', ax=axes[0])
    axes[0].set_title(f'Espectrograma — RUIDOSO: {noisy_name}')
    fig.colorbar(img0, ax=axes[0], format='%+2.0f dB')

    img1 = librosa.display.specshow(S_wien, sr=sr, x_axis='time',
                                    y_axis='log', cmap='viridis', ax=axes[1])
    axes[1].set_title(f'Espectrograma — WIENER: {out_name}')
    fig.colorbar(img1, ax=axes[1], format='%+2.0f dB')

    img2 = librosa.display.specshow(S_ref, sr=sr, x_axis='time',
                                    y_axis='log', cmap='viridis', ax=axes[2])
    axes[2].set_title(f'Espectrograma — LIMPO: {clean_name}')
    fig.colorbar(img2, ax=axes[2], format='%+2.0f dB')

    plt.tight_layout()
    plt.show()


# ==========================================================================
# TABELA FINAL — MÉDIA DE TODOS OS ÁUDIOS
# ==========================================================================
if resultados:
    df = pd.DataFrame(resultados)
    medias = df.drop(columns=['Áudio']).mean()

    linhas_media = [
        (nome, medias[col_r], medias[col_p])
        for nome, col_r, col_p in ORDEM_METRICAS
    ]
    imprimir_tabela(
        f"MÉDIA GERAL — {len(df)} áudio(s) processado(s)",
        linhas_media
    )


# ==========================================================================
# ZIP + DOWNLOAD
# ==========================================================================
if caminhos_salvos:
    zip_path = "/content/denoised_wavs.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in caminhos_salvos:
            zf.write(p, arcname=os.path.basename(p))
    print(f"\n📦 ZIP gerado com {len(caminhos_salvos)} arquivo(s): {zip_path}")
    try:
        files.download(zip_path)
    except Exception as e:
        print(f"[Aviso] Download automático falhou: {e}")
else:
    print("\n[AVISO] Nenhum áudio denoised foi salvo.")

Output hidden; open in https://colab.research.google.com to view.